# AutoLearnMeds — Bootstrap

Run **once per Colab session**. After it succeeds, copy the printed `update_ssh_config.sh` line into a terminal on your Mac, then connect via VSCode Remote-SSH to the host alias `autolearnmeds-colab`.

Required: Colab Pro+ runtime, A100 GPU, **Background execution enabled**.

## Why two phases?

`google.colab.drive.mount()` and `google.colab.auth.authenticate_user()` only work from a notebook cell (they need the IPython kernel for the interactive consent dialog). So this cell does the Colab-API steps in Python first, then shells out to `colab_bootstrap.sh` for the pure-shell parts (gcsfuse, git clone, uv sync, Drive→GCS sync, SSH tunnel, daemons).

In [ ]:
# REPLACE these values before running. Keep this notebook private.
import os

# Required
os.environ['AUTOLEARNMEDS_GCS_BUCKET']   = 'gs://auto_learn_meds'
os.environ['AUTOLEARNMEDS_REPO_URL']     = 'https://github.com/n-suman/AutoLearnMeds.git'
os.environ['AUTOLEARNMEDS_SSH_PASSWORD'] = 'cWbDRcUsDzyWkYXRdhno8IzD4yqwTjVl'

# Optional
os.environ['AUTOLEARNMEDS_BRANCH']                 = 'phase-0-plumbing'
os.environ['AUTOLEARNMEDS_DRIVE_GOLDEN_SET_ID']    = '1LNxhx5prZNhgheLMZcPbTbq4KfTZoQdX'
os.environ['AUTOLEARNMEDS_DRIVE_RAW_IMAGES_ID']    = '1I7CSQ35Qhe67bwXQYVlRxfaeXDf9UEVR'

# === Colab-API steps (must run from notebook context) ===
print('[notebook] Mounting Google Drive...')
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

print('[notebook] Authenticating for Google Cloud (gsutil + gcsfuse)...')
from google.colab import auth
auth.authenticate_user()

# === Hand off to bash bootstrap for pure-shell steps ===
BRANCH = os.environ['AUTOLEARNMEDS_BRANCH']
REPO   = os.environ['AUTOLEARNMEDS_REPO_URL'].replace('https://github.com/', '').replace('.git', '')
!curl -sSL https://raw.githubusercontent.com/{REPO}/{BRANCH}/scripts/colab_bootstrap.sh -o /tmp/colab_bootstrap.sh
!bash /tmp/colab_bootstrap.sh

## After the bootstrap finishes

1. Read the cloudflared hostname from the output above.
2. On your **Mac**, in your terminal: `cd /Users/apple/AutoLearnMeds && ./scripts/update_ssh_config.sh <hostname>`
3. In **VSCode**: `Cmd-Shift-P` -> Remote-SSH: Connect to Host -> `autolearnmeds-colab`.
4. From inside VSCode, open a terminal and run `make verify` to confirm the rig is green.

If anything fails, the agent (Claude Code in your VSCode) reads `/tmp/keepalive.log`, `/tmp/sync_to_gcs.log`, and `/tmp/colab_ssh_output.txt` to diagnose.